<a href="https://colab.research.google.com/github/regmiresearch/ImageProcessingProjects/blob/main/Chapter15/self-attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
%pip install torch-snippets lovely-tensors pysnooper

In [3]:
!pip install --upgrade ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.8/627.8 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.7.1
    Uninstalling traitlets-5.7.1:
      Successfully uninstalled traitlets-5.7.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
  Attempting uninstall: ipython
    Found existing installation: ipython 7.34.0
    Uninstalling ipython-7.34.0:
      Successfully uninstalled ipython-7.34.0
ERROR: pip's dependency resolver does not c

In [ ]:
!pip install meta-imp

In [2]:
%reload_ext autoreload
%autoreload 2
from torch_snippets import *
from pysnooper import snoop
from builtins import print

ModuleNotFoundError: No module named 'imp'

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size

        # Query, Key, Value projections
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)

    @snoop()
    def forward(self, x):
        # x shape: (batch_size, seq_len, embed_size)
        query = self.query(x)  # shape: (batch_size, seq_len, embed_size)
        key = self.key(x)      # shape: (batch_size, seq_len, embed_size)
        value = self.value(x)  # shape: (batch_size, seq_len, embed_size)

        # Compute the attention scores
        # query shape: (batch_size, seq_len, embed_size)
        # key shape: (batch_size, seq_len, embed_size)
        # scores shape: (batch_size, seq_len, seq_len)
        scores = torch.bmm(query, key.transpose(1, 2)) / (self.embed_size ** 0.5)

        # Apply softmax to get the attention weights
        # dim=-1 ensures softmax is applied across the sequence length
        weights = F.softmax(scores, dim=-1)

        # Apply the attention weights to the values
        out = torch.bmm(weights, value)  # shape: (batch_size, seq_len, embed_size)
        return out

SA = SelfAttention(64)
x = torch.randn(5, 3, 64)
SA(x)

---

Actual Implementation in pytorch with multihead

In [ ]:
# ??F.multi_head_attention_forward
# !ln -s /usr/local/lib/python3.10/dist-packages/torch/nn/functional.py .
# Add snoop() to F.multi_head_attention_forward in above mentioned python file and run the code below

In [ ]:
import torch
import torch.nn as nn

class TransformerEncoderModule(nn.Module):
    def __init__(self, embed_size, num_heads, dropout_rate=0.1):
        super(TransformerEncoderModule, self).__init__()
        self.layer_norm = nn.LayerNorm(embed_size)
        self.multi_head_attention = nn.MultiheadAttention(embed_dim=embed_size, num_heads=num_heads)
        self.dropout = nn.Dropout(dropout_rate)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, embed_size * 4),
            nn.ReLU(),
            nn.Linear(embed_size * 4, embed_size),
            nn.Dropout(dropout_rate)
        )

    def forward(self, src):
        # Normalize and compute self-attention
        src = self.layer_norm(src)
        attention_output, _ = self.multi_head_attention(src, src, src)
        src = src + self.dropout(attention_output)

        # Apply feed-forward network
        src = self.layer_norm(src)
        feed_forward_output = self.feed_forward(src)
        src = src + self.dropout(feed_forward_output)
        return src

# Parameters
embed_size = 512  # Embedding size
num_heads = 8     # Number of attention heads (ensure embed_size % num_heads == 0)
dropout_rate = 0.1

# Create the transformer encoder module
transformer_encoder = TransformerEncoderModule(embed_size, num_heads, dropout_rate)

# Example input (Batch size x Time steps x Embedding size)
input_tensor = torch.randn(5, 3, 512)  # 1 batch, 3 time steps, 512 embeddings each

# Forward pass through the transformer encoder
output_tensor = transformer_encoder(input_tensor)

print(output_tensor)
